In [8]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict,Literal,Annotated
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage,HumanMessage

In [10]:
generator_llm = ChatOpenAI()
evaluator_llm = ChatOpenAI()
optimizer_llm = ChatOpenAI()

# State
class TweetState(TypedDict):
    topic: str
    tweet: str
    evaluation: Literal["good", "bad"]
    iteration : int
    max_iteration : int


In [ ]:
def generate_tweet(state:TweetState):
    messages = [
        SystemMessage(content="You are a funny and clever assistant that generates tweets."),
        HumanMessage(content=f"""Write a short , original , and hilarious tweet about {state['topic']}.
                    Make sure the tweet is less than 280 characters.
                    Do not use question-answer format.
                    think in meme logic,punchlines or relatable takes.
                     use simple day to day english.
                    """)
    ]
    return {"tweet": generator_llm.invoke(messages).content}

In [12]:
def evaluate_tweet(state:TweetState):
    messages = [
        SystemMessage(content="You are a strict and critical assistant that evaluates the quality of tweets."),
        HumanMessage(content=f"""Evaluate the following tweet on a topic {state['topic']} and rate it as "good" or "bad" based on its humor, originality, and relevance to the topic. 
                    Tweet: {state['tweet']}
                    only return "good" or "bad" as the evaluation result.
                    auto - bad if the tweet is not less than 280 characters,
                    or if it follows a question-answer format, or if it is not in simple day to day english, or if it does not have a meme logic, punchlines or relatable takes.
                    """)
    ]
    evaluation = evaluator_llm.invoke(messages).content.strip().lower()
    return {"evaluation": "good" if "good" in evaluation else "bad"}

In [ ]:
def optimize_tweet(state:TweetState):
    messages = [
        SystemMessage(content="You are a creative and insightful assistant that optimizes tweets based on feedback."),
        HumanMessage(content=f"""The following tweet received a {state['evaluation']} evaluation. 
                    Please provide specific suggestions to improve the tweet's humor, originality, and relevance to the topic {state['topic']}. 
                    Tweet: {state['tweet']}
                    """)
    ]
    return {"tweet": optimizer_llm.invoke(messages).content}

In [ ]:
graph = StateGraph(TweetState)

graph.add_node('generate',generate_tweet)
graph.add_node('evaluate',evaluate_tweet)
graph.add_node('optimize',optimize_tweet)